# Phase 2: Data Understanding
## Profiling the Close File HC Tabs

**CRISP-DM Purpose:** Explore the data sources identified in Phase 1. Profile schema, data types, completeness, quality, and document any issues or surprises found.

**Project:** ML-Driven FP&A Analyst -- Headcount Actuals vs. Forecast Analysis

**Scope:** This notebook profiles the 5 HC tabs in the Close File (HC, HC-US, HC-India, HC-RD, HC-Colleen) to establish a ground-truth understanding of the data we will work with.

---

## 1. Close File Schema

All 5 HC tabs share an identical column layout across **26 columns (A-Z)**.

### Column Map

| Column(s) | Content | Data Source |
|-----------|---------|-------------|
| B | Department name | Static labels (Adaptive hierarchy) |
| C-N | **12 months of trailing actuals** (currently Mar-2025 through Feb-2026) | Adaptive "GAAP Actuals-Month" via Office Connect |
| O | *Blank spacer* | -- |
| P | **Current month actuals** (Mar-2026) | Adaptive "GAAP Actuals-Month (current-2)" |
| Q | **Latest Forecast** for current month (Mar-2026) | Adaptive "Month (current-2)" = Latest Forecast version |
| R | **Month Variance** = P - Q (Actuals - Forecast) | Calculated |
| S | *Blank spacer* | -- |
| T | **Month Variance Commentary** (free text) | Manual entry by FP&A analysts |
| U | *Blank spacer* | -- |
| V | **CWM full-year** (FY2026) | Adaptive CWM version |
| W | **Latest Forecast full-year** (FY2026) | Adaptive Latest Forecast version |
| X | **EOY Forecast Variance** = V - W (CWM - Latest Forecast) | Calculated |
| Y | *Blank spacer* | -- |
| Z | **EOY Variance Commentary** (free text) | Manual entry by FP&A analysts |

### Rolling Window Behavior

The file does **not** grow or manually shift each month. Row 2 contains **parameterized Adaptive version references** (e.g., `GAAP Actuals-Month (current-14)`, `GAAP Actuals-Month (current-13)`, etc.) that Office Connect resolves dynamically. When the report date is advanced one month and the sheets are refreshed, all columns auto-shift to show the new 12-month trailing window. No manual column insertion or deletion is needed.

### Known Issue: HC-US Tab Stale EOY Labels

| Tab | EOY Fiscal Year | Forecast Version Label |
|-----|----------------|------------------------|
| HC | FY2026 | Latest Forecast |
| HC-US | **FY2025** | **2025 10+2** |
| HC-India | FY2026 | Latest Forecast |
| HC-RD | FY2026 | Latest Forecast |
| HC-Colleen | FY2026 | Latest Forecast |

**Confirmed stale.** The HC-US tab's EOY section references FY2025 and a prior forecast version ("2025 10+2"). Its CWM value of 1,030.28 matches Dec-2025 actuals, confirming it's looking at FY2025 rather than FY2026. This needs to be updated to FY2026 / Latest Forecast to be consistent with the other tabs. Flagged for the FP&A team.

## 2. Row Structure: Department Taxonomy

Each HC tab has **162 department rows** (rows 6-169 in HC/HC-US; rows 6-173 in India/RD/Colleen) organized in a 3-level hierarchy.

### Row Types

| Type | Example | Count | Description |
|------|---------|-------|-------------|
| L0 Total | `Entrata` | 1 | Grand total (row 6) |
| L1 Cost Category | `COR`, `S&M`, `R&D`, `G&A` | 4 | Rolls up sub-departments |
| L2 Department | `Engineering`, `Sales`, `IT` | 20 | Operational department |
| L3 Entity Split | `Engineering - US`, `Engineering - India` | 80 | One per entity per department |
| `(Only)` Placeholder | `COR Only (Only)`, `Engineering (Only)` | 25 | Adaptive structural artifacts, **always zero** |
| Summary | `Total OPEX`, `Total Entrata`, `HC excl. LC` | 3 | Bottom-of-sheet aggregates |
| `COR Only` sub-rows | `COR Only - US`, `COR Only - Digital Marketing` | 9 | Mapped to COR but always zero HC |

### Summary Row Position Differences

**Critical for automated parsing:** Summary rows sit at different row numbers across the tabs due to extra placeholder rows in India/RD/Colleen.

| Row Label | HC | HC-US | HC-India | HC-RD | HC-Colleen |
|-----------|-----|-------|----------|-------|------------|
| Total OPEX | 166 | 166 | 170 | 170 | 170 |
| Total Entrata | 167 | 167 | 171 | 171 | 171 |
| HC excl. LC | 169 | 169 | 173 | 173 | 173 |

**Implication:** Any automated reader must search by row label (`column B`) rather than relying on fixed row numbers.

## 3. Data Types and FTE Logic

### Type Distribution

| Type | Row Count | Description |
|------|-----------|-------------|
| Integer (whole HC) | 86 | Standard headcount -- each employee = 1 |
| Decimal (FTE) | 6 | Hours-based FTE from Leasing Center US only |
| All Zero | 70 | Entity splits with no presence + `(Only)` placeholders |

### Which Rows Carry Decimals

Only **6 rows** ever contain decimal values:
- `Leasing Center` (total)
- `Leasing Center - US`
- `Entrata` (L0 total)
- `COR` (L1 total)
- `Total Entrata` (summary)
- `HC excl. LC` (summary -- but LC is added back, so this is Total minus LC, and still picks up the decimal via subtraction)

Decimals propagate **upward only** from `Leasing Center - US` through its parent rollup chain.

### Leasing Center FTE: Dual Logic

| Entity | HC Method | Data Type | Source |
|--------|-----------|-----------|--------|
| **US** | Hours-based FTE calculation | Decimal | ADP scheduled hours → Google Sheet → Adaptive manual entry |
| **India** | Standard headcount (1 per employee) | Integer | Same as all other India employees via Domo/Adaptive import |

India Leasing Center employees are full-time employees counted the same way as all other Entrata India staff. **Only US LC** uses the special hours-to-FTE conversion.

### India LC History

India LC headcount appeared in **Jun-2025** and has grown steadily:

| Month | India LC |
|-------|----------|
| Mar-May 2025 | 0 |
| Jun-Nov 2025 | 10 |
| Dec-2025 | 13 |
| Jan-2026 | 3 |
| Feb-2026 | 43 |
| Mar-2026 | 42 |

Commentary references "+3 Buffer Contact Center Professionals added," suggesting this is an offshore expansion of the contact center function.

## 4. Entity Rollup Validation

### Rule: Total = US + India + RD + Colleen

Each employee maps to exactly one entity. The HC Total tab should equal the sum of the four entity sub-tabs for every department and every month.

### Results

| Test | Scope | Result |
|------|-------|--------|
| Feb-2026 (all department rows) | 28 key departments | **ALL PASS** |
| Mar-2026 (all department rows) | 28 key departments | **ALL PASS** (after file refresh) |
| Mar-2026 (summary rows using correct positions) | Total Entrata | **PASS** (2,130.74 = 1,029.74 + 992 + 75 + 34) |

**Note:** Before the file was refreshed for March, the entity rollup showed a 103.74 discrepancy in the Total tab's Leasing Center row. This resolved after the Office Connect refresh, confirming that **timing of the refresh is critical** -- any analysis on a partially-refreshed file will show false integrity errors.

## 5. Variance Formula Verification

### Month Variance (Column R = Column P - Column Q)

Verified for all major rollup rows in the HC Total tab:

| Department | Actuals (P) | Forecast (Q) | Variance (R) | Check |
|------------|------------|--------------|-------------|-------|
| Entrata | 2,130.74 | 2,145.24 | -14.50 | PASS |
| COR | 791.74 | 803.24 | -11.50 | PASS |
| S&M | 247 | 249 | -2 | PASS |
| R&D | 828 | 829 | -1 | PASS |
| G&A | 264 | 264 | 0 | PASS |
| Total Entrata | 2,130.74 | 2,145.24 | -14.50 | PASS |

**Interpretation:** Negative variance = actuals below forecast (fewer HC than planned). Entrata is 14.5 FTE under forecast for March, driven primarily by COR (-11.5).

### EOY Forecast Variance (Column X = Column V - Column W)

| Department | CWM (V) | Latest Fcst (W) | Variance (X) | Check |
|------------|---------|-----------------|-------------|-------|
| Entrata | 2,434.54 | 2,431.54 | +3 | PASS |
| COR | 956.64 | 950.64 | +6 | PASS |
| S&M | 300.4 | 298.4 | +2 | PASS |
| R&D | 879.6 | 889.6 | -10 | PASS |
| G&A | 297.9 | 292.9 | +5 | PASS |
| Total Entrata | 2,434.54 | 2,431.54 | +3 | PASS |

**Interpretation:** Positive EOY variance = CWM projects higher year-end HC than the latest forecast. R&D is the notable exception at -10, driven by capacity hire reductions "as per JT's direction."

## 6. Month-over-Month Trend Analysis

### Total Entrata HC (trailing 13 months)

| Month | Total HC | MoM Change |
|-------|----------|------------|
| Mar-2025 | 2,001.80 | -- |
| Apr-2025 | 2,007.00 | +5.20 |
| May-2025 | 2,017.27 | +10.27 |
| Jun-2025 | 2,057.49 | +40.22 |
| Jul-2025 | 2,074.45 | +16.96 |
| Aug-2025 | 2,060.39 | -14.06 |
| Sep-2025 | 2,061.85 | +1.46 |
| Oct-2025 | 2,049.61 | -12.24 |
| Nov-2025 | 2,073.05 | +23.44 |
| Dec-2025 | 2,087.28 | +14.23 |
| Jan-2026 | 2,109.36 | +22.08 |
| Feb-2026 | 2,128.50 | +19.14 |
| Mar-2026 | 2,130.74 | +2.24 |

**Observations:**
- Overall upward trajectory from ~2,002 to ~2,131 over 13 months (+6.4%)
- Jun-2025 saw the largest single-month jump (+40), likely from LC India onboarding (10 new) plus organic growth
- Aug and Oct showed slight dips, possibly seasonal attrition
- Mar-2026 growth nearly flat (+2.24), suggesting the company is approaching a steady state or seasonal hiring slowdown

### Leasing Center Volatility

LC is the most volatile department due to its hours-based FTE methodology. Notable swing: **Sep-2025 to Oct-2025 dropped 11.5%** (140.85 → 124.61). This likely reflects seasonal call volume shifts rather than actual headcount changes.

## 7. Variance Commentary Analysis

18 department rows carry commentary for the March close. Commentary falls into distinct categories:

### Commentary Categories

| Category | Count | Examples |
|----------|-------|----------|
| **Department swizzle** | 4 | "Ashley Carson transferred into US not RD", "Asiya Shaikh tagged to Eng instead of IT" |
| **Terminations** | 2 | "India term Roshan Paul", "Dania Urrutia same day term" |
| **Capacity hire adjustments** | 3 | "-4 Capacity Hires for 2026 reduced as per JT's direction" |
| **Backfills / new positions** | 4 | "+1 Brandon Arnold backfill", "+1 New Deal Desk Analyst" |
| **Structural / mapping** | 3 | "Added HC Shweta to provide detail", "-1 T. Holland req#3253 s/b Digital Mkt request ADP change" |
| **Mix / methodology** | 1 | "Alorica/Domestic mix. Forecasted higher Alorica spend." |
| **Clean** | 1 | "Clean" (IT-US, no variance) |

### Pattern: Commentary Lives at Entity-Split Level

Commentary is written at the **Level 3 entity-split** level (e.g., `Engineering - India`, `Finance - US`), not the rolled-up department level. This makes sense because variance explanations typically involve a specific employee action in a specific entity.

### Data Quality Note

Some rows have **zero variance but still carry commentary** (e.g., Implementation-US month variance = 0, but has a swizzle note). This occurs when offsetting movements net to zero within the month but still warrant explanation. Our future analysis should surface commentary regardless of whether the net variance is zero.

## 8. Data Quality Summary

### Checks Performed

| Check | Result | Action |
|-------|--------|--------|
| Entity rollup (Total = US + India + RD + Colleen) | **PASS** (all months, all depts) | None -- integrity confirmed |
| COR Only rows always zero | **PASS** (all 9 COR Only rows, all months, all tabs) | None |
| `(Only)` placeholder rows always zero | **PASS** (all 25 rows) | Safe to filter out in analysis |
| Month Variance formula (R = P - Q) | **PASS** (all tested rows) | None |
| EOY Variance formula (X = V - W) | **PASS** (all tested rows) | None |
| Data completeness (no nulls in actuals) | **PASS** (C-N fully populated for all key depts) | None |
| HC-US EOY version label | **FAIL** -- shows FY2025/"2025 10+2" | Flag for manual fix; exclude US EOY from automated analysis until corrected |
| Pre-refresh entity rollup (Mar-2026) | **FAIL before refresh, PASS after** | Confirms automated analysis must run on fully-refreshed file |

### Zero-Value Rows (70 total)

70 rows are perpetually zero across all months and all tabs. These fall into two groups:
1. **Entity splits with no presence** (e.g., `Training - India` = 0 because Training is US-only)
2. **Adaptive `(Only)` structural placeholders** (e.g., `Engineering (Only)` = 0 always)

These should be **filtered out** in any analytical processing to reduce noise. They carry no analytical signal.

## 9. Confirmed Decisions (Phase 2)

| # | Decision | Confirmed By |
|---|----------|--------------|
| D14 | HC-US EOY labels (FY2025/"2025 10+2") are **stale** and need updating to FY2026/Latest Forecast | Manager |
| D15 | India LC employees are **standard full-time HC** (integer, 1 per employee), unlike US LC which uses hours-based FTE | Manager |
| D16 | Phase 1 docs need updating: LC is **US + India** (not US-only) since Jun-2025 | Manager |
| D17 | Office Connect's parameterized version references (`current-N`) handle the rolling window automatically -- no manual column shifting needed | Manager |
| D18 | `(Only)` rows are Adaptive structural artifacts, always zero, safe to filter out | Data profiling |
| D19 | Summary rows are at different positions across tabs (166-169 for HC/US, 170-173 for India/RD/Colleen) -- must search by label not row number | Data profiling |
| D20 | Analysis must run on a **fully refreshed** Close File; partially refreshed files produce false integrity errors | Data profiling |

## 10. Phase 2 Status

| Deliverable | Status |
|-------------|--------|
| Close File schema documented | Done |
| Row structure and department taxonomy profiled | Done |
| Data types and FTE logic documented (including India LC distinction) | Done |
| Entity rollup validated | Done |
| Variance formulas verified | Done |
| MoM trend analysis | Done |
| Commentary patterns cataloged | Done |
| Data quality checks complete | Done |
| Manager checkpoint questions answered | Done |

**Next step:** Phase 3 -- Data Preparation (to be initiated upon manager approval)

Phase 3 will:
- Build a Python parser that reads the Close File by label (not row number) to handle the tab-specific row offsets
- Flatten the 5-tab structure into a single normalized DataFrame (department × entity × month)
- Filter out the 70 zero-value rows and `(Only)` placeholders
- Separate the LC US FTE values from standard integer HC for type-aware processing
- Create the analytical dataset for variance decomposition